<a href="https://colab.research.google.com/github/weagan/SSM-and-Mamba/blob/main/Simple_State_Space_Model_(without_Mamba)_for_Shakespeare2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Simple State Space Model (Mamba) for Shakespeare Text Generation
Supports CPU, single GPU, or multi-GPU training

A true Mamba architecture goes far beyond this by making its state-space parameters input-dependent (selective) and by using hardware-aware parallelization for its scan operation, allowing it to achieve performance comparable to Transformers for long sequences while maintaining linear scaling.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import requests
from typing import Optional
import time # Import the time module

# =============================================================================
# CONFIGURATION
# =============================================================================

class Config:
    # Model hyperparameters
    d_model = 256          # Hidden dimension size
    n_layers = 4           # Number of Mamba layers
    vocab_size = None      # Will be set based on data

    # Training hyperparameters
    batch_size = 128        # Batch size for training (increased from 32)
    seq_length = 512       # Sequence length for training (increased from 128)
    learning_rate = 3e-4   # Learning rate
    num_epochs = 5         # Number of training epochs

    # Data configuration
    data_fraction = 0.01    # Fraction of the dataset to use (e.g., 0.1 for 10%)

    # Device configuration
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    use_multi_gpu = torch.cuda.device_count() > 1

config = Config()

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

class ShakespeareDataset(Dataset):
    """Dataset for Shakespeare text with character-level tokenization"""

    def __init__(self, text: str, seq_length: int):
        """
        Args:
            text: Raw text data
            seq_length: Length of each training sequence
        """
        self.seq_length = seq_length

        # Create character-level vocabulary
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)

        # Create mappings between characters and indices
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}

        # Encode entire text as integers
        self.data = [self.char_to_idx[ch] for ch in text]

    def __len__(self):
        """Number of sequences in dataset"""
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        """
        Get a single training example
        Returns:
            x: Input sequence [seq_length]
            y: Target sequence [seq_length] (shifted by 1)
        """
        # Get sequence of indices
        x = torch.tensor(self.data[idx:idx + self.seq_length], dtype=torch.long)
        y = torch.tensor(self.data[idx + 1:idx + self.seq_length + 1], dtype=torch.long)
        return x, y

def load_shakespeare_data():
    """Download and load Shakespeare dataset"""
    print("Downloading Shakespeare dataset...")
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    response = requests.get(url)
    text = response.text
    print(f"Loaded {len(text)} characters")
    return text

# =============================================================================
# SIMPLE MAMBA BLOCK (Simplified State Space Model)
# =============================================================================

class MambaBlock(nn.Module):
    """
    Simplified Mamba/SSM block
    This is a minimal implementation focusing on the core concepts
    """

    def __init__(self, d_model: int):
        """
        Args:
            d_model: Model dimension
        """
        super().__init__()
        self.d_model = d_model

        # State space parameters (simplified)
        # A: State transition matrix
        # B: Input-to-state matrix
        # C: State-to-output matrix
        self.A = nn.Parameter(torch.randn(d_model, d_model) * 0.01)
        self.B = nn.Linear(d_model, d_model)
        self.C = nn.Linear(d_model, d_model)

        # Input and output projections
        self.input_proj = nn.Linear(d_model, d_model)
        self.output_proj = nn.Linear(d_model, d_model)

        # Activation
        self.activation = nn.SiLU()

    def forward(self, x):
        """
        Forward pass through Mamba block
        Args:            x: Input tensor [batch, seq_len, d_model]
        Returns:
            Output tensor [batch, seq_len, d_model]
        """
        batch, seq_len, d = x.shape

        # Project input
        x_proj = self.input_proj(x)

        # Initialize hidden state
        h = torch.zeros(batch, self.d_model, device=x.device)

        # Process sequence step by step (this is the SSM recurrence)
        outputs = []
        for t in range(seq_len):
            # Get current input
            x_t = x_proj[:, t, :]

            # State space update: h_{t+1} = A * h_t + B * x_t
            h = torch.matmul(h, self.A.T) + self.B(x_t)
            h = self.activation(h)

            # Output: y_t = C * h_t
            y_t = self.C(h)
            outputs.append(y_t)

        # Stack outputs
        output = torch.stack(outputs, dim=1)

        # Final projection
        output = self.output_proj(output)

        return output

# =============================================================================
# FULL MAMBA MODEL
# =============================================================================

class SimpleMamba(nn.Module):
    """
    Simple Mamba language model
    Stacks multiple Mamba blocks with residual connections
    """

    def __init__(self, vocab_size: int, d_model: int, n_layers: int):
        """
        Args:
            vocab_size: Size of vocabulary
            d_model: Model dimension
            n_layers: Number of Mamba layers
        """
        super().__init__()
        self.d_model = d_model

        # Token embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Stack of Mamba blocks
        self.layers = nn.ModuleList([
            MambaBlock(d_model) for _ in range(n_layers)
        ])

        # Layer normalization
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_layers)
        ])

        # Output head
        self.output_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        """
        Forward pass
        Args:
            x: Input token indices [batch, seq_len]
        Returns:
            Logits [batch, seq_len, vocab_size]
        """
        # Embed tokens
        x = self.embedding(x)  # [batch, seq_len, d_model]

        # Pass through Mamba layers with residual connections
        for layer, norm in zip(self.layers, self.layer_norms):
            # Pre-norm + residual connection
            x = x + layer(norm(x))

        # Final normalization and output projection
        x = self.output_norm(x)
        logits = self.lm_head(x)  # [batch, seq_len, vocab_size]

        return logits

# =============================================================================
# TRAINING LOOP
# =============================================================================

def train_model(model, dataloader, optimizer, criterion, device, num_epochs):
    """
    Train the Mamba model
    Args:
        model: Model to train
        dataloader: DataLoader for training data
        optimizer: Optimizer
        criterion: Loss function
        device: Device to train on
        num_epochs: Number of epochs
    """
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        num_batches = 0
        start_epoch_time = time.time() # Start time for the epoch

        for batch_idx, (x, y) in enumerate(dataloader):
            start_batch_time = time.time() # Start time for the batch

            # Move data to device
            x = x.to(device)
            y = y.to(device)

            # Forward pass
            logits = model(x)  # [batch, seq_len, vocab_size]

            # Reshape for loss computation
            # Loss expects [batch * seq_len, vocab_size] and [batch * seq_len]
            logits = logits.view(-1, logits.size(-1))
            y = y.view(-1)

            # Compute loss
            loss = criterion(logits, y)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Update weights
            optimizer.step()

            # Track loss
            total_loss += loss.item()
            num_batches += 1

            end_batch_time = time.time() # End time for the batch
            batch_time = end_batch_time - start_batch_time

            # Print progress every 50 batches
            if (batch_idx + 1) % 50 == 0:
                avg_loss = total_loss / num_batches
                print(f"Epoch [{epoch+1}/{num_epochs}], "
                      f"Batch [{batch_idx+1}/{len(dataloader)}], "
                      f"Loss: {avg_loss:.4f}, "
                      f"Batch Time: {batch_time:.4f}s") # Print batch time

        end_epoch_time = time.time() # End time for the epoch
        epoch_duration = end_epoch_time - start_epoch_time

        # Print epoch summary
        avg_loss = total_loss / num_batches
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Average Loss: {avg_loss:.4f}, Duration: {epoch_duration:.2f}s\n") # Print epoch duration

# =============================================================================
# TEXT GENERATION
# =============================================================================

def generate_text(model, dataset, prompt: str, max_length: int, device, temperature: float = 1.0):
    """
    Generate text using the trained model
    Args:
        model: Trained model
        dataset: Dataset (for char mappings)
        prompt: Starting text
        max_length: Maximum length to generate
        device: Device to run on
        temperature: Sampling temperature (higher = more random)
    Returns:
        Generated text
    """
    model.eval()

    # Encode prompt
    input_ids = [dataset.char_to_idx[ch] for ch in prompt]
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

    generated = prompt

    with torch.no_grad():
        for _ in range(max_length):
            # Get logits for last position
            logits = model(input_ids)
            logits = logits[0, -1, :] / temperature

            # Sample from distribution
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Append to sequence
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

            # Decode and add to generated text
            next_char = dataset.idx_to_char[next_token.item()]
            generated += next_char

            # Keep only last seq_length tokens to prevent memory issues
            if input_ids.size(1) > config.seq_length:
                input_ids = input_ids[:, -config.seq_length:]

    return generated

# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    print("=" * 70)
    print("MAMBA SHAKESPEARE TEXT GENERATION")
    print("=" * 70)

    # Check device configuration
    print(f"\nDevice: {config.device}")
    if torch.cuda.is_available():
        print(f"GPU Count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Multi-GPU: {config.use_multi_gpu}\n")

    # Load data
    text = load_shakespeare_data()

    # Apply data fraction
    if 0.0 < config.data_fraction < 1.0:
        original_length = len(text)
        text = text[:int(original_length * config.data_fraction)]
        print(f"Using {config.data_fraction*100:.0f}% of data: {len(text)} characters (original: {original_length})")

    dataset = ShakespeareDataset(text, config.seq_length)
    config.vocab_size = dataset.vocab_size
    print(f"Vocabulary size: {config.vocab_size} characters\n")

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=0  # Use 0 for Colab compatibility
    )

    # Create model
    print("Initializing model...")
    model = SimpleMamba(
        vocab_size=config.vocab_size,
        d_model=config.d_model,
        n_layers=config.n_layers
    )

    # Move model to device
    model = model.to(config.device)

    # Wrap in DataParallel if multiple GPUs available
    if config.use_multi_gpu:
        print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    # Setup training
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    criterion = nn.CrossEntropyLoss()

    # Train model
    print("Starting training...\n")
    train_model(model, dataloader, optimizer, criterion, config.device, config.num_epochs)

    # Generate text
    print("\n" + "=" * 70)
    print("GENERATING TEXT")
    print("=" * 70 + "\n")

    # Unwrap model if using DataParallel
    generate_model = model.module if config.use_multi_gpu else model

    prompts = [
        "ROMEO:",
        "To be or not to be",
        "The king"
    ]

    for prompt in prompts:
        print(f"Prompt: '{prompt}'")
        generated = generate_text(
            generate_model,
            dataset,
            prompt,
            max_length=200,
            device=config.device,
            temperature=0.8
        )
        print(f"Generated:\n{generated}\n")
        print("-" * 70 + "\n")

if __name__ == "__main__":
    main()

MAMBA SHAKESPEARE TEXT GENERATION

Device: cuda
GPU Count: 1
  GPU 0: Tesla T4
Multi-GPU: False

Loaded 1115394 characters
Using 1% of data: 11153 characters (original: 1115394)
Vocabulary size: 58 characters

Initializing model...
Model parameters: 1,347,130

Starting training...

Epoch [1/5], Batch [50/84], Loss: 2.7850, Batch Time: 2.9078s
Epoch [1/5] completed, Average Loss: 2.5148, Duration: 248.16s

Epoch [2/5], Batch [50/84], Loss: 1.8423, Batch Time: 2.9072s
Epoch [2/5] completed, Average Loss: 1.7685, Duration: 248.60s

Epoch [3/5], Batch [50/84], Loss: 1.4757, Batch Time: 2.9124s
Epoch [3/5] completed, Average Loss: 1.4042, Duration: 248.11s

Epoch [4/5], Batch [50/84], Loss: 1.1632, Batch Time: 3.1117s
Epoch [4/5] completed, Average Loss: 1.1210, Duration: 248.97s

Epoch [5/5], Batch [50/84], Loss: 0.9715, Batch Time: 2.9242s
Epoch [5/5] completed, Average Loss: 0.9324, Duration: 248.29s


GENERATING TEXT

Prompt: 'ROMEO:'
Generated:
ROMEO:,
Where hoves, now are a smy lion e